# Cross-source name resolver

Build a resolver that maps a canonical UFCStats fighter name to the correct identifier for each external source. It was built for Wikipedia and Fight Matrix, and for Google Trends before Trends was dropped from the axis. Earlier verification showed each source represents names differently: Wikipedia has false positives and disambiguation suffixes, and Fight Matrix flips non-Western name order. A basic string match fails on both.

**Approach:**
1. Direct match (handles the majority)
2. Token-sort match (catches name-order flips like Yan Xiaonan / Xiaonan Yan)
3. Manual alias table (known overrides for edge cases)
4. MMA relevance check (Wikipedia only; rejects false positives like the Shane Bannon mismatch surfaced in testing)

**Input:** the fighter roster from UFCStats (the unique fighters from fights_per_fighter.parquet)

**Output:** a resolution table mapping each fighter to their Wikipedia title and Fight Matrix name, with a status per source (matched / unmatched / alias / no_article)

## Section 1: Setup and load the canonical roster

In [ ]:
# BLOCK 1: Mount Drive and Library Import

import pandas as pd
import numpy as np
import re
from datetime import datetime
from pathlib import Path
import json

# data location (portable across Drive and a local repo checkout)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except (ImportError, ModuleNotFoundError):
    pass  # not in Colab

DRIVE_DIR = Path('/content/drive/MyDrive/Masters in Artificial Intelligence Applied to Sport/'
                 'Masters Final Project/Pugnator mapper valorem/EDA/Code Outputs')
OUTPUT_DIR = DRIVE_DIR if DRIVE_DIR.exists() else Path('./data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

USER_AGENT = "UFC-Value-Mapper/0.1 (MSc academic project; contact via GitHub th1555)"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Name resolver run: 2026-07-16 10:37:00


In [ ]:
# BLOCK 2: Load canonical fighter names

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

USER_AGENT = "UFC-Value-Mapper/0.1 (MSc academic project; contact via GitHub th1555)"

BASE_URL = "https://raw.githubusercontent.com/th1555/ufc-fighter-value-mapper/refs/heads/main/raw_data/"
df_results = pd.read_csv(BASE_URL + 'ufc_fight_results.csv')
for col in df_results.select_dtypes(include='object').columns:
    df_results[col] = df_results[col].str.strip()

# every fighter named in a bout
names = set()
for bout in df_results['BOUT'].dropna():
    parts = str(bout).split(' vs. ')
    if len(parts) == 2:
        names.add(parts[0].strip())
        names.add(parts[1].strip())
canonical_names = sorted(names)
print(f"Loaded {len(canonical_names):,} unique fighters from fight_results BOUT column")

print(f"\nSample: {canonical_names[:10]}")

Loaded 2,714 unique fighters from fight_results BOUT column

Sample: ['AJ Cunningham', 'AJ Dobson', 'AJ Fletcher', 'Aalon Cruz', 'Aaron Brink', 'Aaron Phillips', 'Aaron Pico', 'Aaron Riley', 'Aaron Rosa', 'Aaron Simpson']


## Section 2: The resolver core

Three matching functions: source agnostic and they work on string pairs. The source specific logic (Wikipedia API calls, relevance checks) covers all in Section 3.

In [ ]:
# BLOCK 3: Matching functions

def normalise_name(name):
    return ' '.join(str(name).lower().strip().split())
# Try an exact match against target names
def direct_match(canonical, target_names):
    canon_norm = normalise_name(canonical)
    for t in target_names:
        if normalise_name(t) == canon_norm:
            return t
    return None

# Sort name tokens alphabetically, catches name order flips
def token_sort_match(canonical, target_names):
    canon_sorted = ' '.join(sorted(normalise_name(canonical).split()))
    for t in target_names:
        target_sorted = ' '.join(sorted(normalise_name(t).split()))
        if canon_sorted == target_sorted:
            return t
    return None

# Manual overrides lookup
def alias_match(canonical, alias_table):
    key = normalise_name(canonical)
    if key in alias_table:
        return alias_table[key], True
    return None, False

# Layered resolution
def resolve_name(canonical, target_names, alias_table=None):
    # Layer 0: alias table
    if alias_table:
        result, found = alias_match(canonical, alias_table)
        if found:
            return result, 'alias'
    # Layer 1: direct match
    result = direct_match(canonical, target_names)
    if result:
        return result, 'direct'
    # Layer 2: token sort match
    result = token_sort_match(canonical, target_names)
    if result:
        return result, 'token_sort'
    return None, 'unmatched'

# Test
test_targets = ['Xiaonan Yan', 'Rose Namajunas', 'Tatiana Suarez']
for name in ['Yan Xiaonan', 'Rose Namajunas', 'Someone Unknown']:
    resolved, method = resolve_name(name, test_targets)
    print(f"  '{name}' -> '{resolved}' ({method})")

  'Yan Xiaonan' -> 'Xiaonan Yan' (token_sort)
  'Rose Namajunas' -> 'Rose Namajunas' (direct)
  'Someone Unknown' -> 'None' (unmatched)


## Section 3: Manual overrides

Manual overrides for known edge cases. This table grows over time; entries added as demanded.

Only fighters with confirmed problems need entries.

In [ ]:
# BLOCK 4: Manual table

# Populated from verification findings.

# Convention:
#   'wikipedia': 'Article Title'  > use this title
#   'wikipedia': None             > confirmed no article exists
#   'fight_matrix': 'Name As FM'  > use this name for FM matching
#   'trends': 'Query String'      > use this for Trends queries


ALIAS_TABLE = {
    # Verification finding: Wikipedia resolved to Shane Bannon (wrong person)
    'shauna bannon': {
        'wikipedia': None,  # no correct article found; mark as unresolvable
    },
    # Verification finding: no Wikipedia article exists
    'alexia thainara': {
        'wikipedia': None,
    },
}

print(f"Alias table: {len(ALIAS_TABLE)} entries")
for k, v in ALIAS_TABLE.items():
    print(f"  {k}: {v}")

Alias table: 2 entries
  shauna bannon: {'wikipedia': None}
  alexia thainara: {'wikipedia': None}


## Section 4: Wikipedia title resolution with relevance check

The Wikipedia path has both a false positive (wrong person) and no article problem. The resolver uses the MediaWiki opensearch endpoint to check the article intro for MMA relevance before accepting a match.

In [ ]:
# BLOCK 5: Wikipedia resolution functions

MEDIAWIKI_API = "https://en.wikipedia.org/w/api.php"

# Search Wikipedia for article titles matching a fighter name.
def wiki_opensearch(name, user_agent, timeout=10):
    params = {
        'action': 'opensearch', 'search': name,
        'limit': 3, 'namespace': 0, 'format': 'json',
    }
    try:
        r = requests.get(MEDIAWIKI_API, params=params,
                         headers={'User-Agent': user_agent}, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        return data[1] if len(data) > 1 else []
    except Exception as e:
        return []

# Intro Check
def wiki_check_relevance(title, user_agent, timeout=10):
    params = {
        'action': 'query', 'prop': 'extracts', 'titles': title,
        'exintro': 1, 'explaintext': 1, 'format': 'json', 'redirects': 1,
    }
    try:
        r = requests.get(MEDIAWIKI_API, params=params,
                         headers={'User-Agent': user_agent}, timeout=timeout)
        r.raise_for_status()
        pages = r.json()['query']['pages']
        page = next(iter(pages.values()))
        intro = page.get('extract', '').lower()
        return any(kw in intro for kw in ['mixed martial', 'mma', 'ufc', 'fighter', 'fighting'])
    except:
        return False

# End to end resolution
def resolve_wikipedia(canonical_name, alias_table, user_agent):

    # Check alias table first
    key = normalise_name(canonical_name)
    if key in alias_table and 'wikipedia' in alias_table[key]:
        override = alias_table[key]['wikipedia']
        if override is None:
            return None, 'alias_no_article'
        return override, 'alias_override'

    # Opensearch
    candidates = wiki_opensearch(canonical_name, user_agent)
    if not candidates:
        return None, 'no_candidates'

    # Check top candidate for relevance
    top = candidates[0]
    if wiki_check_relevance(top, user_agent):
        return top, 'matched'

    # Top candidate failed relevance; try others
    for alt in candidates[1:]:
        if wiki_check_relevance(alt, user_agent):
            return alt, 'matched'

    return None, 'wrong_person'

print("Resolver ready.")

Resolver ready.


## Section 5: Run the resolver on a sample

Test on comparison batch (women's strawweight), then on a bigger sample.

In [ ]:
# BLOCK 6: Resolve the thin slice sample (regression test)

# Expected from the earlier verification:
#   Wikipedia: 8/10 correct, Shauna Bannon > wrong person (alias block),
#             Alexia Thainara > no article (alias block)
#   Fight Matrix: Yan Xiaonan > 'Xiaonan Yan' (token sort )

thin_slice_fighters = [
    'Rose Namajunas', 'Ashley Yoder', 'Gloria de Paula', 'Tatiana Suarez',
    'Shauna Bannon', 'Jessica Penne', 'Amanda Ribas', 'Alexia Thainara',
    'Denise Gomes', 'Yan Xiaonan',
]

# Fight Matrix
fm_sample_names = [
    'Weili Zhang', 'Mackenzie Dern', 'Virna Jandiroba', 'Tatiana Suarez',
    'Gillian Robertson', 'Xiaonan Yan', 'Tabatha Ricci', 'Denise Gomes',
    'Amanda Lemos', 'Iasmin Lucindo', 'Lupita Godinez', 'Tecia Pennington',
    'Alexia Thainara', 'Jingnan Xiong', 'Amanda Ribas',
]

print("FIGHT MATRIX RESOLUTION")
print()
fm_results = []
for fighter in thin_slice_fighters:
    resolved, method = resolve_name(fighter, fm_sample_names, alias_table=None)
    fm_results.append({'fighter': fighter, 'fm_name': resolved, 'fm_method': method})
    print(f"  {fighter:20s} -> {str(resolved):20s} ({method})")

print()
print("WIKIPEDIA RESOLUTION")
print()
wiki_results = []
for fighter in thin_slice_fighters:
    title, status = resolve_wikipedia(fighter, ALIAS_TABLE, USER_AGENT)
    wiki_results.append({'fighter': fighter, 'wiki_title': title, 'wiki_status': status})
    print(f"  {fighter:20s} > {str(title):30s} ({status})")
    time.sleep(0.5)  # polite pacing

FIGHT MATRIX RESOLUTION

  Rose Namajunas       -> None                 (unmatched)
  Ashley Yoder         -> None                 (unmatched)
  Gloria de Paula      -> None                 (unmatched)
  Tatiana Suarez       -> Tatiana Suarez       (direct)
  Shauna Bannon        -> None                 (unmatched)
  Jessica Penne        -> None                 (unmatched)
  Amanda Ribas         -> Amanda Ribas         (direct)
  Alexia Thainara      -> Alexia Thainara      (direct)
  Denise Gomes         -> Denise Gomes         (direct)
  Yan Xiaonan          -> Xiaonan Yan          (token_sort)

WIKIPEDIA RESOLUTION

  Rose Namajunas       > Rose Namajunas                 (matched)
  Ashley Yoder         > Ashley Yoder                   (matched)
  Gloria de Paula      > Gloria de Paula                (matched)
  Tatiana Suarez       > Tatiana Suarez                 (matched)
  Shauna Bannon        > None                           (alias_no_article)
  Jessica Penne        > Jessica P

In [ ]:
# BLOCK 7: Combined resolution table for the sample

df_fm = pd.DataFrame(fm_results)
df_wiki = pd.DataFrame(wiki_results)

df_resolved = df_fm.merge(df_wiki, on='fighter')

print("Combined resolution table (thin-slice sample):")
print(df_resolved.to_string(index=False))
print()

# Summary counts
print("Fight Matrix resolution:")
print(df_resolved['fm_method'].value_counts().to_string())
print()
print("Wikipedia resolution:")
print(df_resolved['wiki_status'].value_counts().to_string())

Combined resolution table (thin-slice sample):
        fighter         fm_name  fm_method      wiki_title      wiki_status
 Rose Namajunas            None  unmatched  Rose Namajunas          matched
   Ashley Yoder            None  unmatched    Ashley Yoder          matched
Gloria de Paula            None  unmatched Gloria de Paula          matched
 Tatiana Suarez  Tatiana Suarez     direct  Tatiana Suarez          matched
  Shauna Bannon            None  unmatched            None alias_no_article
  Jessica Penne            None  unmatched   Jessica Penne          matched
   Amanda Ribas    Amanda Ribas     direct    Amanda Ribas          matched
Alexia Thainara Alexia Thainara     direct            None alias_no_article
   Denise Gomes    Denise Gomes     direct    Denise Gomes          matched
    Yan Xiaonan     Xiaonan Yan token_sort     Yan Xiaonan          matched

Fight Matrix resolution:
fm_method
unmatched     5
direct        4
token_sort    1

Wikipedia resolution:
wiki_statu

## Section 6: Scale to the full roster (optional, time-dependent)

Running Wikipedia resolution on all 4,000ish fighters would take hours (API rate limits). Running Wikipedia resolution on all 4,000ish fighters would take hours (API rate limits). For this verification, the thin slice regression test is sufficient. The full roster resolution runs later as a batch job.

Instead, this block looks to resolve Fight Matrix names for the full roster using the offline matching layers (no API calls, instant).

In [ ]:
# BLOCK 8: Roster to Fight Matrix resolution (offline, instant)

# This uses only the layered string matching (direct + token-sort),
# not API calls, so it runs in seconds on the full roster.

print(f"Resolving {len(canonical_names):,} canonical fighters against FM sample ({len(fm_sample_names)} names)")
print()

fm_full_results = []
for fighter in canonical_names:
    resolved, method = resolve_name(fighter, fm_sample_names)
    if resolved:  # only record matches to keep the output manageable
        fm_full_results.append({'fighter': fighter, 'fm_name': resolved, 'method': method})

print(f"Matched: {len(fm_full_results)} fighters")
df_fm_full = pd.DataFrame(fm_full_results)
if len(df_fm_full) > 0:
    print(f"\nBy method:")
    print(df_fm_full['method'].value_counts().to_string())
    print(f"\nToken-sort matches (name-order flips caught):")
    token_matches = df_fm_full[df_fm_full['method'] == 'token_sort']
    if len(token_matches) > 0:
        for _, row in token_matches.iterrows():
            print(f"  {row['fighter']} -> {row['fm_name']}")
    else:
        print("  (none in this sample)")

Resolving 2,714 canonical fighters against FM sample (15 names)

Matched: 14 fighters

By method:
method
direct        11
token_sort     3

Token-sort matches (name-order flips caught):
  Xiong Jingnan -> Jingnan Xiong
  Yan Xiaonan -> Xiaonan Yan
  Zhang Weili -> Weili Zhang


## Section 7: Save the resolver and alias table

Save the resolution table and the alias table so downstream notebooks can import them. The resolver functions themselves will eventually live in a shared module (`resolver.py`) so the later notebooks can import them.

In [ ]:
# BLOCK 9: Save resolution sample and alias table

df_resolved.to_csv(OUTPUT_DIR / 'name_resolution_sample.csv', index=False)
with open(OUTPUT_DIR / 'name_alias_table.json', 'w') as f:
    json.dump(ALIAS_TABLE, f, indent=2)
print(f"Saved resolution sample and alias table to {OUTPUT_DIR}")

Saved resolution sample and alias table to /content/drive/MyDrive/Masters in Artificial Intelligence Applied to Sport/Masters Final Project/Pugnator mapper valorem/EDA/Code Outputs


## Summary

Resolver tested on 10 thin-slice fighters and 2,714 full roster names.

This notebook develops the resolver design and runs it as a thin-slice regression test on ten fighters. The production Wikipedia resolution for the full active roster, with its fuller set of hand-verified overrides, runs in notebook 13; the figures below are the thin-slice test, not the full-roster resolution.

**Wikipedia:** 8 of 10 matched; 2 recorded as alias entries with no valid article (Shauna Bannon, Alexia Thainara). No false positives accepted.

**Fight Matrix:** 4 direct matches and 1 token-sort match from the thin-slice sample. The full roster resolved against the 15-name Fight Matrix sample found 14 matches (11 direct, 3 token-sort). All three token-sort catches were Chinese name-order flips (Xiong Jingnan, Yan Xiaonan, Zhang Weili). Expect more when running against all Fight Matrix divisions later.

**Alias table:** 2 manual entries, both recording fighters with no reliably resolvable Wikipedia article (Shauna Bannon, whose name resolves to the wrong person and is rejected by the relevance check; Alexia Thainara, who has no article). At thin-slice scale a small manual table is a safe floor, since it introduces no false positives. It does not scale to the full roster, where cases would go undiscovered; a principled replacement (for example, Wikidata identifiers or a confidence-scored match with human review of low-confidence cases) is future work.

**Outputs:** name_resolution_sample.csv, name_alias_table.json